# Assignment: Post-Training an LLM to Reason

This assignment has three required parts, in order, plus optional
"above and beyond" work:

1. **Part 1 -- GRPO.** You build and train the full SFT + GRPO pipeline
   yourself, on a real task (below), and evaluate what GRPO actually
   bought you over SFT alone. **No trained checkpoint is provided at any
   point in this assignment -- every checkpoint you evaluate, you
   trained.**
2. **Part 2 -- GRPO alternatives (required).** Once Part 1 is done, you
   train and evaluate exactly one alternative to GRPO of your choice
   (cheaper, RL-adjacent methods) against the exact same task and held-out set, and
   compare it to your own Part 1 results. Part 2 below gives the scope,
   the three method scripts you run, the shared collapse checks, and how
   to submit it.
3. **Part 3 -- build your own SFT data (required, in stages).** You
   critique the given SFT data, build your own, and then work through
   three further stages; Stages 1 and 2 each end in a decision you record
   about whether to continue, because a full run costs hours of GPU. Stopping
   honestly at a decision point is a legitimate outcome.

Every part below states explicitly what's **given** to you vs. what
**you build** -- if something isn't listed as given, assume you're
building it.

**How it is graded, in one line.** **Two thirds of the grade** comes
from what we specifically ask for: Part 1 (100 points), Part 2 (50) and
Part 3 (50). **One third** comes from "above and beyond" ideas that we do
**not** guide you on, graded on novelty, importance, an explained rationale and good
experimental process, even if the experiment fails or refutes your
hypothesis (a well-designed failure earns a lot of credit). The full
tables are under "Grading" in Part 1 (Part 1's own 100), in each part's
submission section, and in "Extras" at the end.


## Files that come with this assignment

Everything below is in the same folder as this notebook (the links open the files on GitHub).

- Setup: [`requirements.txt`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/requirements.txt)
- Optional companion deck: [`P1_Preflight.ipynb`](P1_Preflight.ipynb), "Before you spend GPU"
- Given Part 1 helpers, `course_lib/`: [`course_lib/__init__.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/__init__.py), [`course_lib/callbacks.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/callbacks.py), [`course_lib/crn.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/crn.py), [`course_lib/eval_harness.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/eval_harness.py), [`course_lib/formatting.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/formatting.py), [`course_lib/grpo_loop.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/grpo_loop.py), [`course_lib/policy_checks.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/policy_checks.py), [`course_lib/rl_readiness.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/rl_readiness.py), [`course_lib/sft_skeleton.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/sft_skeleton.py)
- Given pieces for this task, `course_lib/bs_reasoning/`: [`course_lib/bs_reasoning/__init__.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/bs_reasoning/__init__.py), [`course_lib/bs_reasoning/expr_harness.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/bs_reasoning/expr_harness.py), [`course_lib/bs_reasoning/reward.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/bs_reasoning/reward.py), [`course_lib/bs_reasoning/rule_extract.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/bs_reasoning/rule_extract.py), [`course_lib/bs_reasoning/scenarios.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/bs_reasoning/scenarios.py), [`course_lib/bs_reasoning/sft_examples.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/bs_reasoning/sft_examples.py), [`course_lib/bs_reasoning/sft_format_check.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/course_lib/bs_reasoning/sft_format_check.py)
- Part 2 scripts, `part2_scripts/`: [`part2_scripts/self_distill.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/part2_scripts/self_distill.py), [`part2_scripts/opd_train.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/part2_scripts/opd_train.py), [`part2_scripts/rlsd_train.py`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/part2_scripts/rlsd_train.py), [`part2_scripts/README.md`](https://github.com/kenperry-public/ML_Advanced_Fall_2026/blob/master/assignments/rft_alternatives/student/part2_scripts/README.md)

<!--- #include (requirements.txt) --->
<!--- #include (course_lib/__init__.py) --->
<!--- #include (course_lib/callbacks.py) --->
<!--- #include (course_lib/crn.py) --->
<!--- #include (course_lib/eval_harness.py) --->
<!--- #include (course_lib/formatting.py) --->
<!--- #include (course_lib/grpo_loop.py) --->
<!--- #include (course_lib/policy_checks.py) --->
<!--- #include (course_lib/rl_readiness.py) --->
<!--- #include (course_lib/sft_skeleton.py) --->
<!--- #include (course_lib/bs_reasoning/__init__.py) --->
<!--- #include (course_lib/bs_reasoning/expr_harness.py) --->
<!--- #include (course_lib/bs_reasoning/reward.py) --->
<!--- #include (course_lib/bs_reasoning/rule_extract.py) --->
<!--- #include (course_lib/bs_reasoning/scenarios.py) --->
<!--- #include (course_lib/bs_reasoning/sft_examples.py) --->
<!--- #include (course_lib/bs_reasoning/sft_format_check.py) --->
<!--- #include (part2_scripts/self_distill.py) --->
<!--- #include (part2_scripts/opd_train.py) --->
<!--- #include (part2_scripts/rlsd_train.py) --->
<!--- #include (part2_scripts/README.md) --->


## Before you start

This assignment assumes you have seen the Intro series and the first
sessions of the RL for LLMs series (RLLM-1 to RLLM-3: the verifier, the
reward model, the reference model and the KL penalty). RLLM-1 to RLLM-3
do **not** teach group-relative advantage: this notebook's own paragraph
"GRPO in one paragraph", below, is the source you can rely on, and the
Intro's C2 session is the longer treatment. The paragraphs below restate,
in this notebook, every other idea Part 1 leans on, so nothing in Part 1
sends you to a slide deck you may not have. Read them once now; later
sections use these words without re-defining them.

**Words this assignment uses.**

- **Brief**: the scenario text the model reads (for example, "Spot is
  82.38, strike 70.87 ..."). The **prompt** is the more general word: the
  brief plus a fixed system instruction, which is everything the model is
  conditioned on.
- **Group** and **$G$**: for each prompt GRPO samples $G$ completions
  from the current policy; those $G$ completions are the group.
- **Gate**: a check you run before spending more effort or compute on the
  next step. In this assignment "gate" means only the *suggested*
  pre-flight checks (the exploration check and the readiness sweep, plan
  steps 2 and 3a). The checks that judge a trained policy are called the
  **collapse checks**, named A to D under "Checking whether a policy
  collapsed" below; checks A and B are the two **required checks**
  `evaluate.py` must run at the end of Part 1 (plan step 4).
- **Money plot**: the headline comparison the whole task builds toward,
  *mean reward* for each of four policies (base model, SFT-only, GRPO,
  oracle) on the same held-out scenarios, in one figure or table in your
  report. Its per-skill version is the accuracy panel (plan step 4).
- **Oracle**: a reference policy that is handed the scenario's *true*
  $\sigma$ and $r$ directly, with no extraction step. You do not train it;
  it is the ceiling the other policies are compared against.
- **Ladder stage**: one step of the pipeline in the order you run it
  (SFT, then the signal stage, then the full stage). Each stage writes
  its own log.
- **Held-out**: scenarios the model never trained on, drawn from a
  separate template split, used only for evaluation. Everyone uses the
  same ones: `make_scenarios(200, seed=1, held_out=True)` (200 scenarios,
  seed 1), so results are comparable across students and across parts.

**SFT teaches shape, not correctness.** In supervised fine-tuning (SFT)
you collect (prompt, ideal response) pairs and train with the ordinary
next-token cross-entropy loss, on the response tokens only. At each
response token the model conditions on the *true* preceding response
tokens, not on whatever it would have generated itself. That is called
**teacher forcing**: the model is always shown the correct path, token by
token. SFT is therefore **off-policy** in the SFT-versus-RL sense: the
training examples were never generated by this model under its own
distribution; they are an external target it is forced to imitate. (This
is a different use of "off-policy" from the Intro's behavior-policy
versus target-policy sense.) The consequence is that SFT can only teach
what is represented in its examples. Even a *correct* worked example does
not guarantee the model learns *how to produce* a correct answer to a new
problem; it may learn only the *shape* of a correct one: the tags, the
step-by-step layout, plausible-looking lines. This is **recitation versus
computation**: does the output look like reasoning, or does it actually
reason? SFT is still essential: it establishes the shape a reward
function must be able to parse before it can score anything. Its ceiling
is structural: more or better examples do not fix it.

**Reward function versus reward model.** A reward *model* (classic
RLHF) is a separate neural network trained on human comparisons to
predict a preference score; RL then runs against that learned score,
which is only as good as the noisy preferences it learned from. A reward
*function* is a deterministic program that scores a completion directly,
for tasks where correctness is programmatically checkable (math, code
that can be run, format compliance). The score is exactly right, not
"probably right according to a learned model", and it needs no training
and no labels. A rule-based reward function of this kind is what the
Intro series calls a **verifier**. The catch: it only works where a
checkable structure exists. This is the recipe behind current
reasoning-model RL; the reward here is the verifiable-reward flavor, not
RLHF.

**GRPO in one paragraph.** For a prompt: sample $G$ completions from the
*current* policy at temperature above zero (so they differ); score each
with the reward function, giving $r_1, \ldots, r_G$; compute each
completion's advantage relative to its own group, $A_i = (r_i - \bar r) /
(\mathrm{std}(r) + \epsilon)$ (here $\mathrm{std}$ is the group's
population standard deviation, and $\epsilon$ is a tiny constant, $10^{-4}$
in the given loop, that keeps the division finite when a group's rewards
are all equal); then take a policy-gradient step that
pushes probability up for above-average completions and down for
below-average ones, in both directions, for every group; and usually add a
KL penalty against a frozen reference model. GRPO never needs an absolute
quality bar, only *some* variance within a group, which is why it can
start from a policy that is still mostly wrong.

**No variance, no signal.** The advantage divides by the group's own
standard deviation. If all $G$ completions get the *same* reward, there is
nothing to compare and nothing is pushed in either direction. That happens
for real reasons: the reward function cannot parse any of the completions,
or the task is trivially right (or trivially wrong) for every sample.
Either way GRPO has nothing to learn from until that changes, which is why
it is worth knowing, before a full run, whether your policy's own samples
produce a *range* of rewards at all.

**A quieter failure: a good score from a policy that ignores the prompt.**
There is also a quiet version. Real variance exists from group to group,
training runs, the aggregate reward climbs, and the policy still is not
doing what you wanted. If the SFT starting point never gave the model a
reason to condition on the prompt, GRPO can only amplify whatever
prompt-*independent* regularity scores well on average. A policy that
always answers with the same safe, middling guess can clear a generous
reward threshold on many prompts by chance, and GRPO will reinforce it,
because from its point of view that is what "getting rewarded more" looks
like. So the money-plot ordering (base model worst, SFT-only better, GRPO
best, oracle on top) can be exactly right *and* the policy can have
collapsed: nothing in the number distinguishes "got better at the task"
from "found a better constant guess". The only way to tell is to look at
the outputs: does the answer actually *change* when the prompt does?

**Process credit versus outcome credit.** *Outcome* credit scores only
whether the final answer is right. *Process* credit scores the steps that
led to it (here: how many of the required derivation lines resolve). They
can disagree: a completion can resolve every step and still land on a
wrong price because it extracted a wrong $\sigma$, or get the price right
by luck with broken steps. Reading the two components separately tells you
what the model is actually doing.

**Checking whether a policy collapsed.** Four named checks. Checks A to
C are decoded **greedily** (`do_sample=False`), because real evaluation
decodes greedily: a temperature-sampled probe can look diverse from the
tails of an already-peaked distribution even after the greedy mode has
collapsed.

- **Check A, the behavior check (`check_output_varies`).** It decodes the
  policy on a small held-out probe set chosen to span the scenario space,
  takes the parameters each output implies, and fails if fewer than a
  handful of distinct outputs appear or no parameter spans a meaningful
  fraction of its legal range. Passing means the policy conditions on the
  prompt, at least on this probe; it is not proof of correctness. Its
  min/max test asks whether *any* output differs from *any* other, not
  whether the *typical* output depends on the prompt, so a small
  non-collapsed tail can satisfy it. A real checkpoint from this course
  emitted one value on 90% of held-out probes and still passed, because
  the other 10% spanned a wide range. Passing `require_keys=("sigma", "r")`
  turns on a per-key check that every named key individually varies, plus
  a majority-share check that fails a key whose single most common value
  covers too large a share of probes. A check can have a blind spot: ask
  precisely what it does and does not rule out. **Required in Part 1
  (plan step 4) and in Part 2.**
- **Check B, the ordering check (`assert_money_plot_ordering`).** It
  asserts that the four policies' mean rewards come out in the expected
  order (plan step 4). **Required in Part 1.** In Part 2 it is not
  asserted for the alternative methods (a method that lands below GRPO is
  a finding, not an error); just report where each lands.
- **Check C, the scenario-blind baseline.** A synthetic policy that
  ignores the prompt and always emits one fixed, correctly formatted
  answer, scored on the same reward and held-out set. If a trained
  policy's aggregate matches it closely, that independently confirms
  collapse. **You build and run it in Part 2; Part 1 does not ask for
  it.**
- **Check D, the paired before/after check.** Two policies' decoded
  outputs on the same probe set, compared key by key. **Part 2, at least
  once** (described under "The collapse checks" there).

**Is a gap between two means real?** Two policies scored on the *same*
scenarios should be compared pair by pair, not as independent samples. A
bootstrap on the paired differences gives a 95% interval that does not
assume a bell-shaped distribution; an interval that straddles zero means
the claimed improvement is not backed by your own data. The helper is
`course_lib.policy_checks.paired_significance` (given function #15, below;
Part 2 asks you to use it, and you can use it in Part 1 on any gap you
claim). Part 1 asks you not to claim more than your numbers support.

**Training is itself random, even with a fixed seed.** Identical code,
hyperparameters and SFT checkpoint with a different random seed can give a
genuinely different policy; in this course's own three-seed replicate,
final held-out accuracy ranged from a majority-collapsed policy to one
within a few points of the ceiling. Worse, the *same* seed, rerun in a
fresh GPU session, is not guaranteed to reproduce. `torch.manual_seed`
seeds the random-number streams, but it does not force every GPU
operation to combine parallel work in the same order, and floating-point
addition is not associative. A plain forward pass at a fixed shape on
fixed hardware is normally reproducible; the **backward pass** is the real
exposure, since PyTorch's own documentation names several gradient paths
(flash-attention's backward, `index_add_`, related accumulation ops) as
using nondeterministic atomic operations by default. A tiny per-step
difference compounded over hundreds of steps, each feeding the next, is
enough for "identical" runs to end up in different places: arithmetic, not
corruption. `torch.backends.cudnn.deterministic=True` and
`torch.use_deterministic_algorithms(True, warn_only=True)` narrow this,
at a real cost in speed, with silent fallbacks in `warn_only` mode, and
only on the same hardware and driver stack; use them when you specifically
want to isolate "is this difference my change or just noise", not by
default. The practical rule: before debugging your code after a bad run,
rerun the identical code with a different seed.

**Compute.** Colab is not required, but you need a GPU for the SFT and
GRPO stages: we trained on a T4 and do not recommend CPU for them
(completions are about 134 tokens each, and GRPO generates $G$ of them per
step). We confirmed end to end that a T4-class GPU is sufficient for every
stage, including the full GRPO run. Per-stage times: this text does not
have a measured table for GRPO. The one figure we have is that a 20-step
smoke run (one fixed scenario, $G=4$) took about 2 minutes on a T4;
`run_grpo_stage` logs `step_secs` for every step, so time a few steps and
multiply by your planned step count (plan step 3) before you commit to a
long run.

**Set up the environment (do this first).** Whatever you run on --
Colab's default image included -- don't assume its preinstalled package
versions match what `course_lib` was written against, or even that
everything you need is preinstalled (Colab's default image commonly
doesn't include `trl`). The versions we confirmed work end to end are
**`trl==0.14.0`** and **`transformers==4.48.3`** (our requirements file
also lists `datasets` and `accelerate`, unpinned). Pin `trl` at that
version if you use the collator named in plan step 1:
`DataCollatorForCompletionOnlyLM` is gone from newer `trl` releases, and
an import error on it means you have a newer one. On Colab:

```
!pip install trl==0.14.0 transformers==4.48.3 datasets accelerate
```

then restart the runtime (Runtime -> Restart session) before importing
anything. Do **not** pin `numpy` (the bundle's `requirements.txt` lists
`numpy` without a version, and `datasets` and `accelerate` unpinned too):
Colab's kernel imports numpy at boot, before `pip` runs, and forcing a
different version leaves the kernel fighting a stale in-memory copy. `torch` is hardware-coupled (tied to your GPU/CUDA
setup), so keep whatever your environment already resolves cleanly.

`course_lib` is not a pip package; you import it straight from the
bundle. The **bundle root** is the directory that contains the
`course_lib/` folder. Put that directory on `sys.path` before any
`course_lib` import, in every notebook and script you write:

```
import sys
sys.path.insert(0, "/path/to/bundle_root")   # the folder that contains course_lib/
```

Record the versions you actually used in your submission (see "How to
submit Part 1"): a result nobody can reproduce because nobody recorded
what actually ran isn't a fully reported result. If you prefer other
versions, that is allowed; say which.


## Part 1: GRPO

### The task

Your starting point is a specific, weak, off-the-shelf model:
**`HuggingFaceTB/SmolLM2-135M-Instruct`** -- small enough to train
quickly end to end, and, as you'll confirm yourself before touching any
training (plan step 0, below), not already reliable at this task out of
the box.

**The goal**: teach this model to correctly price a European call option
using the Black-Scholes formula, given a natural-language description of
the trade.

Your model reads a **brief** like:

> *"Spot is 82.38, strike 70.87, 1.34 years to expiry. The desk's marking
> it at 31. r = 0.014."*

(The brief is the scenario text itself. Together with a fixed system
instruction, it forms the full **prompt** the model actually sees --
both words show up throughout this assignment; "brief"
is this task's scenario content specifically, "prompt" is the more
general term for whatever text the model is conditioned on, brief
included.)

The model must respond by explicitly working through the derivation, not
just stating a final number. The response format, precisely:

- A `<think>...</think>` block: the step-by-step derivation, one line at
  a time, each in the form `` name = `expr` ``. The backtick-quoted
  expression on each line is later **executed by a harness**, not just
  pattern-matched -- it has to be real, evaluable arithmetic referencing
  the right prior step names, not free-form text.
- A closing `<answer>...</answer>`, restating the final price. **The
  reward ignores what is inside `<answer>`**: it scores only the executed
  `` name = `expr` `` lines (the `price` line), so even `<answer>1</answer>`
  earns full credit if the chain is right. This matters if you design your
  own completion format in Part 3: the answer tag is for readers, not
  something the reward checks, so do not assume it constrains the model.

Concretely:

```
<think>
sigma = `0.31`
r = `0.014`
d1 = `(log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*sqrt(T))`
d2 = `d1 - sigma*sqrt(T)`
Nd1 = `ncdf(d1)`
Nd2 = `ncdf(d2)`
price = `S*Nd1 - K*exp(-r*T)*Nd2`
</think>
<answer>18.40</answer>
```

That's exactly **seven** required lines, in this order, using exactly
these names -- the harness matches on these names, not a count or a
shape, so a completion using different names for the same quantities
doesn't get credit for them:

1. `sigma`
2. `r`
3. `d1`
4. `d2`
5. `Nd1`
6. `Nd2`
7. `price`

$S$/$K$/$T$ (spot, strike, time-to-expiry) are always given as plain
numbers in the brief; $\sigma$ (volatility) and $r$ (rate) are the two
values your model has to *extract* from the brief's phrasing, which
varies.

Lines 3-7 (`d1`/`d2`/`Nd1`/`Nd2`/`price`) are the **same formula text on
every scenario** -- e.g. `d1`'s expression always references `S`, `K`,
`T`, `sigma`, `r` symbolically, never a scenario's literal numbers baked
into the expression itself. There's nothing scenario-specific for the
model to *derive* in these five lines, only to reproduce correctly and
then have the harness evaluate against whatever `sigma`/`r` you claimed.
Only lines 1-2 (`sigma`, `r`) carry any real per-scenario variation. Keep
this in mind once you're looking at your own per-skill panel -- it
predicts which skill should saturate fast and which one is actually
hard.

Getting this right requires the model to do two genuinely separate
things -- worth checking each on its own, not just the pair together:

1. **Correctly recite the Black-Scholes derivation steps**: the seven
   lines above, in order, each a valid expression referencing the right
   prior names.
2. **Correctly extract $\sigma$/$r$ from the brief's natural-language
   phrasing** -- a different skill from (1): a perfect derivation built
   on the wrong $\sigma$/$r$ still produces a wrong price.

Plan step 0, below, has you examine the *untrained* base model directly
against both of these before you train anything.

This is the running example of "Before you start", made concrete: the
`<think>`/`<answer>` shape is what SFT teaches; getting $\sigma$/$r$
and every derivation step *right* on a scenario the model hasn't seen
before is entirely GRPO's job.


### Reward

A restricted-expression harness re-executes your model's own claimed
derivation lines (only `log`, `sqrt`, `exp`, `ncdf` are legal function
calls) against the brief's true $S$/$K$/$T$ and *your model's own claimed*
$\sigma$/$r$ -- **not** the scenario's true $\sigma$/$r$. A bad
extraction cascades into a bad price exactly the way a bad reading would
on a real desk; the reward doesn't quietly correct it for you.

Reward = 0.3 x process credit + 0.7 x outcome credit. Process credit is
the fraction of the seven canonical lines that resolve (the count of them
is `lines_resolved`, an integer from 0 to 7, returned in the `details`
dict that `reward()` gives back next to the reward; it is not a column of
`metrics.jsonl`). Outcome credit is the final price's accuracy against the
true price, falling linearly from 1 at an exact match to 0 at a relative
error of `PRICE_TOL`. Malformed completions (zero lines resolve) score
$-0.1$, below every valid-but-wrong one.

This reward function is **given** (see below) -- you don't design or
tune it, only train against it. It is a rule-based reward function (what
the Intro series calls a **verifier**), not a trained reward model, and
it's exact, not a learned approximation -- which is only possible because
the completion's correctness is something a program can actually check.

**Process credit versus outcome credit**, as defined in "Before you
start": the "fraction of seven lines that resolve" term is the process
credit; the price-accuracy term is the outcome credit. Think about #4
below asks you to find a case where they disagree.


### Given vs. built

**Given, this task's own pieces (6 files, all under `course_lib/bs_reasoning/`):**

1. `scenarios.py` -- brief generator
2. `expr_harness.py` -- the restricted-expression re-execution harness
3. `rule_extract.py` -- a reference hand-rolled $\sigma$/$r$ extraction
   rule, for comparison only, not something your model is trained to
   imitate
4. `reward.py` -- the reward function described above, weights already
   fixed and measured, not yours to tune
5. `sft_examples.py` -- **your Part 1 SFT training examples.** Unlike
   every other project in this course, and unlike this task's own
   later extension (Part 3, below), you are not designing the
   completion format or the noising strategy yourself for this required
   part -- `make_sft_examples(tokenizer, n=1500, seed=0)` returns the
   `{"prompt", "completion"}` rows directly, already in the exact
   `<think>`/`<answer>` shape shown in "The task" above. Read it before
   you use it -- Part 3 is going to ask what you'd have done differently.
6. `sft_format_check.py` -- `check_no_preamble(completions)`, the
   format-compliance check used in plan step 2 below.

**Also given, general training/eval infrastructure (6 modules, not
specific to this task):**

1. `course_lib/grpo_loop.py` -- a working GRPO training loop
   (`run_grpo_stage`), with the group-relative advantage computation
   already implemented. It automatically writes a
   per-step `metrics.jsonl` to your checkpoint directory -- you don't
   build this logging yourself, just point your plotting code at the
   file it produces.
2. `course_lib/rl_readiness.py` -- `readiness_sweep`, the $G$-sweep
   machinery for the suggested pre-flight in plan step 3a below.
   Separate from, and not used by, `base_rate.py`.
3. `course_lib/sft_skeleton.py` -- the generic cross-project SFT-dataset
   skeleton other projects in this course use. Not what `sft_examples.py`
   is built on (this task's completions aren't JSON, that skeleton
   assumes they are) -- listed here for completeness, not something
   you'll call directly for Part 1.
4. `course_lib/eval_harness.py` -- held-out evaluation scaffolding.
5. `course_lib/policy_checks.py` -- the diversity/ordering checks used
   in plan step 4.
6. `course_lib/formatting.py` -- `format_prompt`, the single shared
   prompt-formatting function every stage (SFT, GRPO, eval) must call
   identically -- see "Given functions" below for why that matters.

**You are not implementing GRPO's training loop or the policy-gradient
math from scratch** -- a working loop, logging and a held-out harness are
real infrastructure cost, and they are what this assignment hands you.
Your job is wiring
this task's specifics (the reward function, the prompt/scenario logic,
the extraction target) into that given infrastructure.

**Built by you (3 required files, plus the suggested `base_rate.py`):**

1. `sft_train.py` (required) -- SFT training driver, trained against the given
   `sft_examples.py` rows.
2. `base_rate.py` -- *suggested, not required* (so it is not one of the
   required files): the exploration check for plan step 2, which
   measures the three skills `sigma`, `rate` (together, extraction
   accuracy) and `chain` (chain-resolution rate).
   Self-contained: samples from your SFT checkpoint and
   checks each completion's own $\sigma$/$r$/chain/price against the
   scenario's true values, using only the given `course_lib/bs_reasoning/`
   pieces above -- no `course_lib.rl_readiness` involved.
3. `grpo_train.py` (required) -- the GRPO driver: wires this task's reward function
   into `course_lib.grpo_loop.run_grpo_stage`. If you run the suggested
   $G$-sweep (plan step 3a) it calls
   `course_lib.rl_readiness.readiness_sweep` first; you choose $G$ either
   way.
4. `evaluate.py` (required) -- held-out evaluation + the per-skill accuracy panel.


### Given functions you'll call directly (15)

Every given file above is documented in its own docstrings -- read those
for exact behavior and edge cases, and for the *why* behind anything that
looks like an unusual choice (`sft_examples.py`'s noising strategy is a
good example: the docstring explains what it's doing and why, right where
the choice lives, not just in this notebook's prose). What follows is
orientation, not a replacement. $S$/$K$/$T$/$\sigma$/$r$ below always
mean this task's own scenario parameters, same as everywhere else in
this document.

1. `course_lib.bs_reasoning.scenarios.make_scenarios(n, seed=0,
   held_out=False)` -> list of scenario objects (`.text` is the brief,
   `.params` the true $S$/$K$/$T$/$\sigma$/$r$). `held_out=True` reserves
   a separate template split for evaluation-only use. **Training data:**
   `make_scenarios(1302, seed=0)` (what we used for the GRPO pool; any
   size and seed with `held_out=False` is fine). **Held-out, the same for
   everyone:** `make_scenarios(200, seed=1, held_out=True)`. Each scenario also
   carries `.vol_bucket`/`.rate_bucket` $\in$ `{"resolvable",
   "adversarial"}` -- how unambiguously the brief's phrasing states
   $\sigma$/$r$. You'll use these two fields directly in plan step 3.
2. `course_lib.bs_reasoning.sft_examples.make_sft_examples(tokenizer,
   n=1500, seed=0)` -> `n` `{"prompt", "completion"}` rows, already in
   the exact format shown in "The task" above -- your Part 1 SFT data
   (plan step 1). Read `sft_examples.py`'s own source, not just this
   summary, if you're going to critique it in Part 3.
3. `course_lib.bs_reasoning.rule_extract.rule_extract_vol(clause)` /
   `rule_extract_rate(clause)` -> the reference hand-rolled extraction,
   for the plan step 0 comparison and "Think about" #3 only.
4. `course_lib.bs_reasoning.expr_harness.evaluate_chain(text,
   prompt_vals)` -> re-executes the completion's own claimed derivation
   lines against `prompt_vals` (the real $S$/$K$/$T$) and whatever
   $\sigma$/$r$ the completion itself claimed.
5. `course_lib.bs_reasoning.expr_harness.chain_extractor(text)` ->
   `(parsed, error)`. Pass this as the `extractor=` argument to
   `readiness_sweep` and `run_grpo_stage` below (#9, #10) -- it's what
   makes their built-in parse-rate logging understand this task's
   `` name = `expr` `` lines instead of assuming JSON. Not a bug, but
   worth knowing: `chain_extractor` evaluates the chain with fixed dummy
   $S$, $K$, $T$ (100, 100, 1), because it only asks whether all seven
   required names resolve, not whether the numbers are right. So the
   values in its `parsed` dict (for example `d1`) differ from what
   `evaluate_chain` gives with the scenario's real values. Use
   `evaluate_chain` for numbers; use `chain_extractor` only for the
   resolved-or-not signal.
6. `course_lib.bs_reasoning.reward.reward(completion_text, params)` ->
   `(reward, details)` -- the reward function itself, described above.
   Also exposes `PRICE_TOL` (0.25) -- the relative tolerance the reward
   itself uses for "close enough." See Rules below for why this matters
   beyond the reward computation itself.
7. `course_lib.formatting.format_prompt(tokenizer, project,
   scenario_text)` -- renders the chat-template prompt for one scenario's
   brief. `make_sft_examples` (#2) already calls this for you at SFT-data
   time; **call it identically yourself at GRPO time and eval time --
   write it once, import it everywhere, never reformat inline.** The
   single most common way this kind of pipeline silently breaks is the
   model being evaluated on a token distribution it never actually
   trained on, because one of the call sites formatted the prompt
   slightly differently.
8. `course_lib.sft_skeleton.make_sft_dataset(scenarios, keys, ranges,
   tokenizer, project, n=1500, seed=0, int_keys=())` -> the generic
   cross-project SFT-dataset skeleton mentioned above -- not what this
   task's `sft_examples.py` is built on, listed for completeness only.
9. `course_lib.rl_readiness.readiness_sweep(sample_fn, reward_fn,
   prompts, candidate_Gs, keys, ranges, extractor=None)` -> `{G: {
   "parse_rate", "err_counts", "within_group_std_mean", "frac_all_tied"}}`
   (what the columns mean: plan step 3a). You supply `sample_fn(prompt, g)` -> list of `g` sampled completion
   strings, and `reward_fn(prompts, completions)` -> list of floats.
   For this task pass `keys=None, ranges=None` and
   `extractor=chain_extractor` (the two arguments are for JSON tasks and
   are ignored when an extractor is given, but they stay required
   positionally). What we passed: `candidate_Gs` = 4, 8, 16, over 8 real
   training prompts, sampling at temperature 1.0 with at most 160 new
   tokens.
10. `course_lib.grpo_loop.run_grpo_stage(sft_ckpt, ckpt_dir,
    prompt_for_step, reward_fn, keys, ranges, *, num_generations,
    max_steps, save_steps, temperature=1.0, beta=0.04,
    learning_rate=5e-6, max_completion_length=48, ..., seed=0,
    extractor=None)` -> the final checkpoint path. **Pass
    `max_completion_length=160`, not the default 48** (see plan step 3).
    Also pass `keys=None, ranges=None, extractor=chain_extractor`. `seed`
    (default 0) seeds torch's random streams and the order of prompts: it
    is the argument to change when a table row says "rerun with a different
    seed". You supply
    `prompt_for_step(rng)` -> one formatted prompt string per step, and
    the same shape `reward_fn` as #9. Writes `metrics.jsonl` itself, one
    line per step: `step, parse_rate, err, reward_mean, within_group_std,
    frac_all_tied, pg_loss, kl, step_secs` -- `parse_rate` here *is* this
    task's chain-resolution rate, since `chain_extractor` (#5) is what's
    driving it.
11. `course_lib.eval_harness.evaluate_policies(scenarios, policies,
    score, m_eval=1024, seed0=12345)` -> tidy `(scenario_id, policy_name,
    reward)` rows, every policy scored against the same seeds.
12. `course_lib.policy_checks.check_output_varies(params_list, ranges,
    min_unique=3, min_relative_range=0.05, require_keys=None)` ->
    `(ok, report)`, and `assert_money_plot_ordering(means, constraints)`
    -> raises with every violated ordering listed, or returns `means`
    unchanged. Both described under plan step 4, below, including the
    exact `params_list` format.
13. `course_lib.bs_reasoning.sft_format_check.check_no_preamble(completions)`
    -> `(ok, violations)`. Not needed for Part 1 -- the given
    `sft_examples.py` rows already satisfy it. Relevant if you build your
    own SFT data in Part 3, below: a diagnostic for whether your own
    design kept or deliberately dropped this property.
14. `course_lib.policy_checks.mean_rewards_from_rows(rows)` -> `{policy:
    mean_reward}`, from the tidy rows `evaluate_policies` (#11) returns.
    Feeds `assert_money_plot_ordering` (#12) and the money plot.
15. `course_lib.policy_checks.paired_significance(rows, policy_a,
    policy_b)` -> a dict with the paired mean difference, a bootstrap 95%
    interval and win/loss/tie counts. First *required* in Part 2
    (see "Is the gap real?" there); available now for any gap you claim
    in Part 1.


### Rules (4)

1. $\sigma$/$r$ must be extracted correctly for the rest of the chain to
   have any chance of being right -- see "Reward" above.
2. $S$/$K$/$T$ are always given as plain numbers; only $\sigma$/$r$ carry
   surface variety in how they're phrased.
3. Every derivation line must be `` name = `expr` `` -- a line the
   harness can't parse simply doesn't count toward the process-credit
   term, it isn't a hard failure of the whole completion.
4. **Tolerance consistency**: wherever you check whether an extracted
   $\sigma$/$r$ (or a resulting price) counts as "correct" -- the
   exploration check (plan step 2) included -- use the reward's own
   `PRICE_TOL` (0.25 relative: for $\sigma$ and $r$ applied to their
   value, for `price` to the price), not an arbitrary stricter standard of
   your own. The reward function doesn't require exact recovery, only
   that the resulting price lands within this tolerance; a stricter check
   than that measures something other than what actually determines
   whether GRPO has a reward signal to learn from, and can make a real
   signal look like a hard zero. (A zero from a strict proxy is not the
   same finding as a zero from the real reward; if you get a suspicious
   zero, test it against the real reward function before concluding
   there is nothing to learn from.)

(Two more rules -- an SFT-data leakage audit, and a `<think>`-block
formatting constraint -- come back in Part 3, once you're building your
*own* SFT examples instead of using the given ones. Both are
unconditionally satisfied by `sft_examples.py`, so they don't apply to
Part 1.)


### The plan

**The four skills.** Whenever this assignment says "skill" it means one of
four things, each judged on one completion against the scenario's true
values (the same definitions our own checks used):

- `sigma`: the completion's own `sigma = ...` line resolves to a value
  within 25% (relative) of the true $\sigma$.
- `rate`: the completion's own `r = ...` line resolves to a value within
  25% (relative) of the true $r$.
- `chain`: all seven required lines resolve (exact names, in the harness),
  whether or not the numbers are right.
- `price`: the resolved `price` line lands within `PRICE_TOL` (0.25
  relative) of the true Black-Scholes price.

"Extraction accuracy" below means `sigma` and `rate`; "chain-resolution
rate" means `chain`. Your `evaluate.py` measures all four skills for the
accuracy panel (plan step 4) whether or not you ran the optional
exploration check. $\sigma$ and $r$ are judged by their *value*; `price`
by the resulting price. 25% is the same number as `PRICE_TOL`; see Rule 4.

0. **Reward/harness sanity check (a plan step; its log file is optional).** Before training anything, run the
   given `rule_extract.py` against a batch of scenarios and confirm it
   extracts correctly (it should -- it's a verified reference). It takes
   one clause at a time, not the whole brief. To get the clauses, split
   the brief into its three sentences:
   `import re; spot, vol_clause, rate_clause = re.split(r"(?<=\.)\s+", sc.text)`.
   A split happens only at a period followed by whitespace, so decimals
   such as `.23` or `82.38` stay whole; on 1,500 training and 1,500
   held-out scenarios this gave exactly three pieces every time. Then
   compare `rule_extract_vol(vol_clause)` with `sc.params.sigma` and
   `rule_extract_rate(rate_clause)` with `sc.params.r`.
   Then hand-inspect a few completions from the *untrained*
   `SmolLM2-135M-Instruct` on the same scenarios, checking both skills
   from "The task" separately: does it produce anything shaped like the
   `<think>`/`<answer>` format at all, with real derivation steps? And,
   independently, does it correctly read $\sigma$/$r$ out of the brief's
   phrasing? (It won't reliably do either -- this is your first real data
   point for the "recitation vs. computation" question in "Before you start".)
   What you see here is what "Think about" 1 asks you to report, so do
   this step even though saving its log (`stage0.jsonl`) is optional. If
   you do not save the log, keep your notes (a few prompts, the base
   model's outputs, what you judged) in a text file or write them directly
   under Think-about 1 in `report.md`; the report is what is read.
1. **SFT.** Train `sft_train.py` against the given
   `sft_examples.make_sft_examples` rows (given function #2, above) --
   for this required part, you're building the trainer, not the data.
   Your choice how to train: HuggingFace/TRL's `SFTTrainer` is fine, and
   so is a hand-rolled training loop -- pick whichever you're more
   comfortable debugging, this isn't graded on which you chose. What it
   must do:
   - **Loss on the response tokens only.** Each training text is the
     rendered prompt followed by the completion; mask everything up to
     the assistant turn so only the completion is trained on (what we
     used with TRL 0.14: `DataCollatorForCompletionOnlyLM` with
     `response_template="<|im_start|>assistant\n"`, on texts built as
     `prompt + completion + tokenizer.eos_token`).
   - **The prompt text comes from `format_prompt`** (given function #7),
     the chat-template function. The given rows already contain the
     rendered `prompt`; never re-render it differently at GRPO or eval time.
   - **Starting points, not requirements.** What we used: learning rate
     2e-5, per-device batch 8 with gradient accumulation 2, 2 epochs over
     roughly 1,250 to 1,500 rows (`make_sft_examples(tokenizer, n=1500,
     seed=0)` returns 1,444), maximum sequence length 320 tokens (the
     longest example was 278 tokens, measured with the end-of-sequence
     token included), float32 on a T4. We chose 2 epochs from a
     dose sweep on our own data; the right dose depends on your rows.
   - **What `sft.jsonl` should contain:** your training log, one JSON
     object per logged step (at least `step` and `loss`), so a reader can
     plot loss against step. The format is yours.
   - **A check we ran afterwards (suggested).** Greedy-decode about 200
     held-out prompts from the SFT checkpoint and count the completions
     in which all seven required lines resolve. We wanted at least 99%
     before moving on, because SFT's job here is the format. It says
     nothing about whether the numbers are right.
2. **Pre-flight, part one: the exploration check, `base_rate.py`
   (strongly suggested, not required).** A GRPO run costs hours of GPU,
   and it can only amplify what the model already does some of the time
   (RL amplifies; it does not invent). So before the full run we checked,
   cheaply, whether the SFT checkpoint ever succeeds at each skill the
   reward needs: sample from the checkpoint (temperature 1, real
   training prompts: temperature 1 is the temperature the GRPO run
   samples at, so it shows the diversity the run will actually see;
   greedy decoding would hide it) and measure extraction accuracy and
   chain-resolution rate (the skills defined above).
   These are two genuinely different measurements (see "The task"
   above), but one sampled batch scored for both is the natural and
   sufficient way to build this (that is what the given
   `course_lib.bs_reasoning` pieces are shaped for). **If either is
   exactly zero, we stopped and reported it rather than patching around
   it**: if your SFT checkpoint's own samples cannot produce *any* range
   of rewards, GRPO has nothing to learn from ("no variance, no signal",
   above) and training past that point just burns compute. This is what
   *we* did and found useful for saving GPU; it is not a claim that it is
   the best possible check, and **you may use other evidence** for the
   same decision (see "Pre-flight and decisions", below). (Don't confuse
   this check with the readiness sweep in step 3a: the exploration check
   asks whether your SFT checkpoint succeeds at each skill at all; the
   sweep helps pick $G$.)
3. **GRPO -- pre-flight part two (suggested), then two required training
   stages.** For both stages, pass `max_completion_length=160` to
   `run_grpo_stage` (full wiring is shown after 3c).

   **3a. Pre-flight, part two: choosing $G$ (the readiness sweep,
   strongly suggested, not required).** $G$ is not handed to you as a
   fixed number, and an inherited $G$ is a risk: the number that suits
   another project's checkpoint and reward can leave your groups with no
   signal at all. Signal grows with $G$ and cost grows linearly, so the
   question is which $G$ is enough for *your* checkpoint and *your*
   reward. What we did: run the **readiness sweep**
   (`course_lib.rl_readiness.readiness_sweep`, signature in "Given
   functions" #9), with candidate $G$s measured against your real SFT
   checkpoint and the real reward, on a slice of real training prompts,
   and read the table it returns. Its three columns, each computed from
   the sampled groups:

   - `parse_rate`: the fraction of sampled completions the extractor can
     parse. A low value means the reward cannot even score many samples.
   - `within_group_std_mean`: for each prompt, the standard deviation of
     its group's rewards; then the mean over prompts. Near zero means the
     groups barely differ, so the advantages are tiny.
   - `frac_all_tied`: the fraction of groups whose rewards are *all
     equal*. Each such group has zero advantage for every member, so it
     contributes no gradient.

   "Ready" is a judgment call you make from the table; no threshold is
   hardcoded anywhere. In our practice we looked for the smallest $G$
   whose groups mostly carried signal (low `frac_all_tied`,
   `within_group_std_mean` clearly above the reward's own noise floor),
   because a larger $G$ costs proportionally more per step. That is what
   we found useful, not a claimed best recipe: **any measure and any
   criterion is acceptable, provided it is tied to your decision and to
   your own numbers.** Other ways to decide include a short pilot run
   watching the first steps, starting at a small $G$ and raising it if
   signal is thin, or an estimate from measured success rates (for a
   binary reward with success probability $p$, the chance a group of $G$
   has both outcomes is $1 - p^G - (1-p)^G$). If you skip the sweep, say
   how you chose $G$ and what evidence supports it; if you ran it, save
   its table (see "How to submit"). An optional companion deck, **"Before
   you spend GPU: a pre-flight"**, walks through this pre-flight and a
   decision record; it is not required for the assignment. It is in this
   folder as `P1_Preflight.ipynb`; read it before your first GPU run
   (before steps 3a and 3b), since that is where it saves you GPU time.

   **3b. Signal stage (required).** Train GRPO first on the easier
   sub-population: scenarios where `vol_bucket == "resolvable"` **and**
   `rate_bucket == "resolvable"` (both fields are on every scenario from
   `make_scenarios`, see "Given functions" #1). Extraction is close to
   unambiguous there, so this stage's reward signal is mostly about
   reliably resolving the arithmetic chain rather than the harder
   judgment call of reading confusable phrasing. Run for roughly 100-150
   steps, and inspect its `metrics.jsonl` before moving on. **Required,
   but not for the reason you might expect**: this stage's trained
   weights are never carried into 3c -- 3c restarts from your SFT
   checkpoint again (see below), so running this stage cannot protect 3c
   from collapsing.
   What it actually buys you: a cheap, early check that your reward/
   harness plumbing produces real signal at all, on the population where
   extraction is closest to unambiguous, *before* you spend the bulk of
   your compute budget on the harder full-pool stage. If this stage's
   `reward_mean` never moves and `within_group_std` stays near zero
   throughout, something in your pipeline may be broken -- find that
   now, not after 800 steps on the full pool. **A caution (our reading,
   not a best recipe):** a flat `reward_mean` over a short run is not by
   itself proof of a bug. In our reference runs the reward sat near 0.5
   until after step 400 to 450 and then either broke out or never did, so
   do not stop a run on reward level at a short step count; use the signal
   columns (`within_group_std`, `frac_all_tied`, `parse_rate`, KL). Save this stage's log as
   `grpo_signal.jsonl`.
   **3c. Full stage.** A fresh GRPO run **from your SFT checkpoint again**
   (not continued from the signal stage's trained weights:
   `run_grpo_stage` always starts from the `sft_ckpt` you pass it), now sampling from the full scenario pool
   -- both resolvable and adversarial buckets -- for roughly 500-800
   steps. This is the checkpoint you evaluate in plan step 4. Save this
   stage's log as `grpo_full.jsonl`.

   `course_lib.grpo_loop.run_grpo_stage` supplies sensible default
   hyperparameters for everything else in both stages (temperature 1.0,
   the KL-to-reference weight `beta=0.04`, learning rate `5e-6`) -- you
   don't need to rediscover those, only to wire in this task's reward
   function, prompt logic, and (for 3b) the bucket filter correctly.

   **The one default you must change: `max_completion_length`.** The
   function's default is 48 tokens, but a full correct seven-line
   derivation is about 134 tokens (measured on the given examples, longer
   completions occur while exploring). Left at 48, every completion is cut
   off before the chain finishes, so almost none resolves: you would expect
   `parse_rate` (the chain-resolution rate) near 0 and
   `within_group_std` near 0 from step 1, with a flat `reward_mean`: it
   looks like a dead pipeline but is a truncation. We passed
   **`max_completion_length=160`** (the same 160 new tokens as in
   evaluation and in the exploration check).

   **What we passed, for both stages** (`G` is your own choice; the step
   counts are ours):

   ```
   # train = make_scenarios(1302, seed=0)   # held_out=False; our GRPO pool
   # scenario_map = {}                      # prompt -> scenario, read by reward_fn
   def prompt_for_step(rng):                # rng: numpy Generator, from run_grpo_stage
       sc = pool[rng.integers(len(pool))]   # pool: see below
       prompt = format_prompt(tok, "bs_reasoning", sc.text)
       scenario_map[prompt] = sc
       return prompt

   def reward_fn(prompts, completions, **kw):
       return [reward(c, scenario_map[p].params)[0]
               for p, c in zip(prompts, completions)]

   run_grpo_stage(sft_ckpt, ckpt_dir, prompt_for_step, reward_fn, None, None,
                  num_generations=G, max_steps=STEPS, save_steps=25,
                  temperature=1.0, beta=0.04, learning_rate=5e-6,
                  max_completion_length=160,
                  extractor=chain_extractor, seed=0)
   ```

   **How the 3b bucket filter plugs in:** it is only the choice of
   `pool`. For 3b, `pool = [sc for sc in train if sc.vol_bucket ==
   "resolvable" and sc.rate_bucket == "resolvable"]` and `STEPS` is about
   150; for 3c, `pool = train` and `STEPS` is about 800. `prompt_for_step`
   must call `format_prompt` (never format the prompt inline), and the
   reward must be computed against the scenario stored for that exact
   prompt. A rerun with a different seed passes `seed=1`.
4. **Evaluation.** Build `evaluate.py` to report a **per-skill** panel
   (`sigma`, `rate`, `chain`, `price`: the four skills defined above),
   not just one aggregate reward number, across four policies:
   - **base model**: the untrained `SmolLM2-135M-Instruct`;
   - **SFT-only**: your SFT checkpoint;
   - **GRPO**: your 3c checkpoint;
   - **oracle**: a policy that returns the completion in the required
     format filled with the scenario's *true* $\sigma$ and $r$ and the
     correctly computed price. It is a benchmark used purely as a ceiling
     for comparison, never training data, so using the true values here is
     allowed (the rule against computing the answer is about training and
     SFT data, not about the benchmark).

   The held-out scenarios are `heldout = make_scenarios(200, seed=1,
   held_out=True)`, the same 200 for everyone. Produce the three model
   policies the same way: render each held-out
   scenario's prompt with `format_prompt`, decode **greedily**
   (`do_sample=False`; we used at most 160 new tokens), and treat the
   decoded text as that policy's completion. Score every policy on the same
   held-out scenarios with `evaluate_policies` (given function #11), whose
   `policies` argument maps a name to a function from scenario to
   completion text, and whose `score` function can simply be
   `lambda sc, completion, seed, m: reward(completion, sc.params)[0]`
   (the given reward). Then `mean_rewards_from_rows(rows)` (given function #14,
   `course_lib.policy_checks`) gives `{policy: mean_reward}`; the panel counts, per skill and policy, how
   many held-out completions hit.

   `evaluate.py` must end with BOTH `course_lib.policy_checks`
   required checks (check A, then check B from "Before you start"),
   behavior first, score second -- check the policy is actually
   conditioning on the prompt before trusting any score built from it:
   1. **Check A:** `check_output_varies` on your GRPO policy's own held-out
      completions, greedy-decoded across at least 10 held-out prompts
      (we used all 200 of the held-out set). "Spanning your scenario pool's
      extremes" means the probe set must include scenarios with the lowest
      and highest true $\sigma$ and $r$ in the pool, not ten similar
      ones; using the whole held-out set satisfies it. For each completion
      pass the $\sigma$, $r$ and price its own derivation resolves to
      (`None` if the chain is incomplete). `params_list` is a list with one
      entry per completion: a dict `{"sigma": ..., "r": ..., "price": ...}`
      of floats, or `None`. We built each entry with
      `resolved = dict(evaluate_chain(c, {"S": sc.params.S, "K": sc.params.K,
      "T": sc.params.T}))` and kept `resolved["sigma"]`, `["r"]`,
      `["price"]` when all three are present. `ranges` is the dict
      `{"sigma": (0.10, 0.50), "r": (0.01, 0.06), "price": (0.0, 150.0)}`
      (the legal ranges we used). A policy can score well
      *on average* while quietly ignoring the prompt -- this is the
      check that catches it, not the aggregate number. **Pass
      `require_keys=("sigma", "r")` explicitly** -- omitting it keeps
      the function's original, coarser behavior (any single key varying
      is enough to pass), which is exactly the gap described under "Checking whether a policy collapsed"
      in "Before you start": a majority-collapsed key
      can still pass if a different key's own diversity keeps the
      joint count up. `require_keys` is what turns on the per-key
      majority-share check that closes it.
   2. **Check B:** `assert_money_plot_ordering` against this task's expected
      ordering, passed as a list of `(better, worse)` pairs (an optional
      third element is a minimum gap) over the policy names:
      ```
      means = mean_rewards_from_rows(rows)
      assert_money_plot_ordering(means, [("grpo", "sft_only"),
                                         ("grpo", "base_model"),
                                         ("oracle", "grpo")])
      ```
      It raises an `AssertionError` listing every violated ordering, or
      returns `means` unchanged.

   **Which required check stops the run.** A failing `check_output_varies`
   does not stop anything: it returns `(False, report)`, print the report
   and discuss it in your report. A failing ordering assertion does stop
   `evaluate.py` (it raises, so the exit is nonzero, loudly rather than a
   CSV you have to eyeball); write the money-plot CSV and the accuracy
   panel *before* calling it so your numbers exist, and then explain in the
   report which ordering failed and why.

   The **money plot** is the headline comparison this whole task builds
   toward: mean reward for each policy (base model, SFT-only, GRPO,
   oracle), on the same held-out scenarios, plotted or tabulated
   together -- the one figure that tells you, at a glance, whether the
   pipeline actually worked. `assert_money_plot_ordering` is a machine
   check on that comparison's *ordering*, so a run that misses its target
   fails loudly instead of producing a money plot you have to eyeball to
   catch the problem.


### Design note: build this to be reused

You'll be evaluating more than one trained policy against this exact
held-out set, in this part and in every part that follows. If you write
`evaluate.py`'s policy-loading and scoring code (load a checkpoint,
greedy-decode a held-out set, score it) as a clean, general function
rather than something wired specifically to a "GRPO checkpoint" variable
name, you'll be able to point that same function at whatever checkpoint
each later part produces, without rewriting it. Not required -- but worth
doing once, well, rather than once per part.


### Think about (these go in your report)

1. **Recitation vs. computation.** Before touching SFT, prompt the *base*
   model (no training at all) to price a few options directly. Does it
   produce formula-shaped text? Does it get the number right? Compare
   this to your SFT-only checkpoint's behavior on the same prompts -- did
   SFT change whether it recites the formula, or only how it's formatted?
2. **Which skill did GRPO actually improve?** Your per-skill panel
   separates extraction accuracy from chain-resolution/price accuracy.
   Did both move, or just one? If only one moved, why might that be --
   is one skill's exploration signal denser than the other's?
3. **The rule-extraction baseline.** The given `rule_extract.py`
   is a five-line heuristic. Where does your trained model's extraction
   accuracy land relative to it? What does the gap (if any) tell you
   about what GRPO exploration can and can't close?
4. **Process vs. outcome credit.** `lines_resolved` (how many of the
   seven lines resolve, from the `details` dict `reward()` returns) and
   price accuracy are two different reward components. Report a case where they
   disagree -- a completion that scores well on one and poorly on the
   other -- and explain what it reveals about the model's behavior.


### Required figures

Your report needs all three:

1. **The per-skill accuracy panel** (`accuracy_panel.png`): `sigma`,
   `rate`, `chain` and `price` accuracy for all four evaluated policies
   (base model, SFT-only, GRPO, oracle) on the held-out scenarios, the
   per-skill version of the money plot. The mean-reward money plot (one
   number per policy, from `mean_rewards_from_rows`) goes in the report
   as a figure or a small table; it is not a separate required file.
2. **One diagnostic plot for your full-pool GRPO stage (3c)**
   (`diagnostics_best.png`): reward, KL-to-reference, and
   chain-resolution rate vs. training step, read from your own
   `grpo_full.jsonl`.
3. **One diagnostic plot for a failed or killed run, with your
   diagnosis** (`diagnostics_failed.png`). Most people hit at least one
   real failure signature along the way. First, what the logged signals in `metrics.jsonl` are:
   `reward_mean` is the mean reward over the step's completions;
   `within_group_std` is the mean over prompts of the standard deviation of
   each group's rewards (near zero means almost no signal);
   `frac_all_tied` is the fraction of groups whose rewards were all equal;
   `kl` is the estimated divergence of the current policy from the frozen
   reference model (the quantity the KL penalty with weight $\beta$
   restrains); `parse_rate` is the fraction of completions the extractor
   can parse, which here is the chain-resolution rate. Then, six
   signatures worth recognizing on sight (the levers are what we would
   try, not guaranteed fixes):

| Symptom | Likely meaning |
|---|---|
| Within-group reward std $\approx 0$ from the start | All $G$ completions score identically -- GRPO has no gradient. Ask why: same temperature? The same output every time regardless of prompt? Something about the reward? This is what the suggested exploration check (plan step 2) is for: it can catch this before you spend a full run on it. |
| Within-group std collapses early, reward stuck | (Reward alone at a short step count is not enough to call this: our reference runs sat on a plateau near 0.5 until after step 400 to 450 -- judge by the within-group std and the other signal columns, our reading.) The policy committed to one output before finding a good one. Try a lower learning rate, a higher $\beta$ (the KL-to-reference weight), or restart from the SFT checkpoint. |
| KL climbs steadily, reward rises then crashes | The policy is drifting far from the reference and degenerating. Raise $\beta$. |
| Chain-resolution rate decays over training | GRPO is destroying the format SFT taught. Raise $\beta$ or lower the learning rate, or restart from the SFT checkpoint (the given reward already scores a malformed completion below every valid-but-wrong one, so the reward is rarely the cause; check what else is being rewarded). |
| Metrics look healthy but each step takes far longer than expected | Something is silently running on the wrong hardware or at the wrong scale -- confirm you're actually running on GPU. |
| Held-out eval looks collapsed or well below the reference numbers, but your training-time metrics looked fine | Don't assume this means your code is broken. `torch.manual_seed` alone does not make a GPU training run fully reproducible -- the identical code with the identical seed can land on a different, worse outcome purely from run-to-run non-determinism (see "Training is itself random, even with a fixed seed" in "Before you start"). **Before debugging your code, rerun the identical code with a different seed.** Collapses again in a similar way -> now you have real evidence of an actual bug, worth digging into (prompt-formatting drift across your SFT/GRPO/eval code is the single most common one). Doesn't collapse the second time -> your original run was very likely just an unlucky draw, not a defect -- report both runs and that conclusion; that's a complete, correct diagnosis, not a shortcoming. |

   A run you killed early because you recognized one of these is a real
   result, not something to hide (but killing on reward level alone at a
   short step count is the mistake the plateau caution warns about) -- report what you saw and the call you
   made.

   **If nothing failed.** Some runs go cleanly. Then say so plainly under
   `## Diagnosis of the failed or killed run` ("none occurred", with the
   evidence that the best run was healthy: read your own
   `diagnostics_best.png`), and leave out `grpo_failed_run.jsonl` and
   `diagnostics_failed.png`, saying in the report that they are absent
   for that reason. No penalty arises from a failure not happening. The
   failed-run diagnosis is worth 15 points (the other 15 of that bucket
   are the Think-about answers). It is for diagnosing a failure signature,
   so with no failure and an honest "none occurred" it can earn only part
   of those 15 (how much depends on how well you read your best run). To
   earn it in full, **induce a failure on purpose** and diagnose it: for example a
   short run of the signal stage (3b) with a changed setting that should
   break it, such as leaving `max_completion_length` at its default 48
   (truncation, see plan step 3), or a learning rate ten times larger.
   Save that run's log as `grpo_failed_run.jsonl`, say in the report that
   the failure was induced and why you expected the symptom you saw, and
   plot it as `diagnostics_failed.png`.


### Pre-flight and decisions (a fixed section of your report)

Before a full GRPO run costs hours of GPU, there are three decisions you
make whether or not you write them down. Your `report.md` must contain a
section with the exact heading **`## Pre-flight and decisions`**, and
under it, for each of the three decisions below, a sub-heading and four
labelled fields. The fixed headings and labels are there so a grader (and
you) can find each entry at a glance.

**The three decisions** (use these exact sub-headings):

- `### Decision 1: Is the SFT checkpoint good enough to start GRPO?`
- `### Decision 2: Which G?`
- `### Decision 3: Go or no-go for the full run`

**The four fields under each** (use these exact labels, bold as shown, one per line):

- `**Decision:**` what you decided, in one sentence.
- `**Criterion:**` what would have made you decide otherwise. State it as a
  rule or threshold you could have applied *before* seeing the outcome
  (for example, "if fewer than 5% of groups carry any signal I would
  raise $G$").
- `**Evidence:**` the file and the numbers, from your own logs
  (`metrics/...jsonl`, a table you computed), that you applied the
  criterion to. A reader should be able to open the file and find the
  number.
- `**Outcome:**` what happened in the full run. If it did not bear your
  decision out, say what you learned.

Add one line at the top of the section, written exactly
`**When:** before the full run` or `**When:** after` (the literal words
`before the full run` or `after`: whether you made these decisions from a
pre-flight before the full run, or reconstructed them afterwards from the
run's own logs). Both are accepted in full. **Retrospective analysis from
`metrics.jsonl` earns full credit; doing the pre-flight before the full
run earns a small bonus**, counted in the additional points below. The
pre-flight is strongly suggested because it saves GPU, but a missing
`readiness_sweep.jsonl` is not penalized.

**Notes on filling the fields.**

- **`When:`** `before the full run` means before the full run *you report
  on*. If your first attempt failed and you re-ran, name which run the
  section is about.
- **Reconstructed criteria.** If you write the section after the run, a
  Criterion may be reconstructed honestly: label it "reconstructed" and
  make it a rule you could have applied from the evidence you had at the
  time (not a threshold chosen to sit just on the right side of your own
  numbers).
- **An example for Decision 1** (good enough to start): "I read 40
  samples at temperature 1; I required that at least a quarter resolve
  all seven lines and that `sigma` land within tolerance in at least 10%
  of them; below that, RL has nothing to amplify."
- **An example for Decision 3** (go or no-go): "Go if Decisions 1 and 2
  held; I would kill the run if, 20 steps in, `within_group_std` were
  still below the reward's noise floor or `frac_all_tied` above 0.8."
- **The reward's noise floor** is the within-group spread you would see
  even if the policy had no real preference, from rounding and sampling
  jitter alone. We did not measure it for this task; one reference point
  we used (our reading, not a measured floor): one more resolved line
  changes the reward by $0.3/7 \approx 0.043$ (the process credit has
  weight 0.3 in the reward, see "Reward", spread over seven lines), so a `within_group_std` far
  below that is hard to tell from none. You may measure your own floor,
  for example from the spread of repeated samples of the same prompt at
  the SFT checkpoint.
- **Sampling temperature is 1** for every pre-flight check, because that
  is the temperature the GRPO run samples at (`run_grpo_stage`'s default),
  so the check sees the diversity the run will see.

**Any measure and any criterion is acceptable** if it is tied to the
decision and to your own numbers. The readiness-sweep columns
(`parse_rate`, `within_group_std_mean`, `frac_all_tied`) are what we
found useful; you can equally use a pilot run, the first steps of
`grpo_signal.jsonl`, an analytic estimate from measured success rates, or
something of your own. What is graded is whether your reasoning is
explicit, checkable against your numbers, and honest about the outcome.
A short worked shape, with invented numbers, to show the form:

```
## Pre-flight and decisions
**When:** before the full run

### Decision 2: Which G?
- **Decision:** G = 8.
- **Criterion:** pick the smallest swept G where fewer than 20% of groups
  are all-tied; if none qualifies, raise G or stop.
- **Evidence:** metrics/readiness_sweep.jsonl: frac_all_tied = 0.62 (G=4),
  0.31 (G=8)... [your own numbers here]
- **Outcome:** grpo_full.jsonl frac_all_tied stayed under 0.3 for 800
  steps; the criterion held. [or: it rose to 0.8 by step 400; I now think ...]
```

(That example is only the shape of an entry, not a target for your
numbers.)


### Grading

The grade has two sources: **two thirds** for what we specifically ask
for (Parts 1 to 3) and **one third** for "above and beyond" ideas we do not
guide you on.

| Part | Points | Where it is described |
|---|---|---|
| Part 1 -- GRPO | 100 | this section |
| Part 2 -- GRPO alternatives | 50 | "Part 2: how to submit and how it is graded" |
| Part 3 -- your own SFT data, staged | 50 | "Part 3: how to submit and how it is graded" |
| Extras ("above and beyond"), one third of the grade | | "Extras" at the end |

**Part 1 is worth 100 points:**

| Bucket | Points | What earns it |
|---|---|---|
| Held-out performance | 20 | the base-model / SFT-only / GRPO / oracle benchmarks, measured honestly with both required checks run first: scored relative to the benchmarks, with no fixed numeric target |
| Evidence for decisions | 30 | your "Pre-flight and decisions" section: for each of the three decisions, a named decision, a stated criterion, evidence a reader can check in your own logs, and the outcome (any measure is acceptable; see above) |
| Diagnosis of a failed run and the Think-about questions | 30 | 15 for your read of a failed or killed run's diagnostics, and 15 for your answers to the four "Think about" questions |
| Report clarity | 20 | the fixed headings, figures that say what they should, claims that match the numbers |

A mediocre final number with a sharp analysis of *why* outscores a good
number you can't explain. "Measured honestly" means all four policies
scored on the same held-out scenarios, both required checks run before the
money plot is shown, and the gain over SFT-only reported as measured. If a
required check fails, that is fine *if your report diagnoses why*.
Submitting a money plot without running both required checks first is not.

**The extras third** (described at the end of this notebook) is for work
beyond what was asked, and you choose it. Doing the pre-flight *before* the
full run earns a small bonus of up to 10 points, counted inside the extras
third.


### How to submit Part 1

**Archive layout (this is the one definition; Part 2 and Part 3 refer
back to it).** Two archives. `part1.tar.gz` holds Part 1 *and* Part 3,
because Part 3 builds on Part 1's pipeline and you write one `report.md`
and keep your logs in `metrics/` folders: Part 1 logs go in
`report/part1/metrics/`, Part 3 logs in `report/part3/metrics/`, and Part 2
logs in its own `metrics/` folder inside `part2.tar.gz`. Its
top level is:

```
part1.tar.gz
  report/
    part1/                    # report.md, figures, metrics/ ... (listed below)
    part3/                    # Part 3 files, listed under "How to submit Part 3"
      sft_data_generator.py   #   your SFT data generator (also copy it into code/)
      sft_data_sample.jsonl   #   a sample of its rows
      sft_eval.md             #   new-checkpoint evaluation, checks A and C output, comparison with the given-data SFT checkpoint
      figures/                #   Part 3 figures (any filenames, referenced from report.md)
      metrics/                #   Part 3 pilot and full-run logs (jsonl; e.g. grpo_signal_p3.jsonl, grpo_full_p3.jsonl)
      panel_part3.csv         #   IF YOU RAN STAGE 3: accuracy panel
      paired_significance.txt #   IF YOU RAN STAGE 3: the paired_significance output
  code/                       # all your source code for Parts 1 and 3
```

`part2.tar.gz` has this top level: `report/part2/` (listed under "Part 2:
how to submit") and `code/` (your Part 2 code).

Part 1's own directory, `report/part1/`:

```
report/part1/
  accuracy_panel.csv        # the per-skill panel, machine-readable
  accuracy_panel.png        # the same panel, plotted
  diagnostics_best.png      # reward/KL/chain-resolution vs. step, your full-pool run (3c)
  diagnostics_failed.png    # same, for the failed/killed (or deliberately induced) run
  metrics/                  # one jsonl log per ladder stage
    sft.jsonl                # your own training log (Trainer or hand-rolled)
    grpo_signal.jsonl        # written automatically by `run_grpo_stage`, plan step 3b
    grpo_full.jsonl          # written automatically by `run_grpo_stage`, plan step 3c
    grpo_failed_run.jsonl    # same, for whatever you killed/diagnosed (see "If nothing failed")
    stage0.jsonl             # IF YOU RAN IT: your log of the step-0 sanity check
    readiness_sweep.jsonl    # IF YOU RAN IT: `readiness_sweep`'s returned table, saved
    exploration_check.json   # IF YOU RAN IT: whatever your `base_rate.py` wrote (any JSON; this filename)
  environment.txt           # pip freeze (or equivalent) from wherever you trained
  report.md                 # fixed headings, see below
```

**`report.md` must contain these headings, spelled exactly** (a grader
checks them mechanically; add whatever else you like under or between
them):

```
## Pre-flight and decisions
## Think about
## Diagnosis of the failed or killed run
## Part 3
## Extras
```

`## Pre-flight and decisions` carries its own fixed sub-headings and
labels (see "Pre-flight and decisions" above). `## Think about` holds your
four answers, numbered 1-4. `## Part 3` holds your staged Part 3 record
(see "Part 3: how to submit and how it is graded"). `## Extras` is
optional: write "None" or leave it out.

**Exact schemas, so this is checkable mechanically, not just by eye:**

- `accuracy_panel.csv`: one row per **(skill, policy)** pair, columns
  `skill,policy,hits,n,accuracy` -- `skill` $\in$ `{chain, sigma, rate,
  price}` (the four skills defined in "The plan" above; `evaluate.py` measures
  them whether or not you ran the optional exploration check), `policy` $\in$ `{base_model, sft_only, grpo,
  oracle}`, `accuracy` is `hits/n` (a fraction in $[0,1]$, not a
  percentage). This is the per-skill panel Required Figure 1 asks for --
  it does not carry mean reward, which stays in "The plan" step 4's
  working `evaluate.py` output rather than a required submission file.
- `readiness_sweep.jsonl` (if you ran the sweep): one line per candidate
  $G$ you swept, each line exactly `readiness_sweep`'s own per-$G$ dict
  (see "Given functions" #9) plus a `"G"` key -- don't invent your own
  fields for this one, just serialize what the given function already
  returns.
- `grpo_signal.jsonl` / `grpo_full.jsonl` / `grpo_failed_run.jsonl`:
  written for you verbatim by `run_grpo_stage` -- see "Given functions"
  #10 for the exact per-line schema. Only `grpo_*.jsonl` is auto-written;
  `stage0.jsonl` and `sft.jsonl` are yours to produce and free-form
  (whatever your chosen trainer logs, or your own print-to-file from the
  step-0 check) -- at minimum, enough to plot loss (or the relevant
  metric) against step.

`readiness_sweep` in particular just returns a Python object in memory,
so save it yourself (`json.dump`) rather than assuming a file appears.
`environment.txt` is your pinned versions from "Before you start" above
-- `pip freeze > environment.txt` (or your environment manager's
equivalent) is enough, run it in whatever environment actually trained
your checkpoints.

Exact filenames matter -- this is what makes grading (and, later, your
own comparisons across Part 2 onward) mechanical rather than a scavenger
hunt. Put your source code (`sft_train.py`, `grpo_train.py`, `evaluate.py`
and the rest) in the top-level `code/` directory shown in the archive
layout above, next to `report/`, not inside `report/part1/`. Your Part 3
files go in `report/part3/` of this same archive, as in that layout (see
Part 3).


### Using AI assistants

You may use AI coding assistants freely for plumbing -- notebook
scaffolding, plotting code, debugging a stack trace, driver scripts. This
assignment is designed so that doesn't shortcut the learning. Three rules
make that true:

1. **Never let an assistant compute the true answer.** Ask one to
   "improve my SFT data" or "check my extraction against the real sigma,"
   and it will cheerfully compute the true value and put it somewhere it
   shouldn't be -- that silently voids the "SFT teaches shape only" claim
   this whole assignment is testing. Most relevant once you're building
   your own SFT examples in Part 3 (Part 1's are given, and already
   audited), but the same risk applies anywhere you touch the reward
   path -- if an assistant produces code you don't understand there,
   that's your risk to catch, not its.
2. **The decisions and their interpretation are yours.** An assistant can
   produce a readiness-sweep table; it can't explain why you chose your
   $G$, what a zero within-group std means for your specific run, or what
   your killed run's diagnostics meant. Those explanations are what's
   graded in the evidence and diagnosis buckets above -- you should be able to walk
   through them without the assistant's help.
3. **Verify library claims empirically.** An assistant will confidently
   describe an API from a version you don't have installed. When
   generated code disagrees with what's actually installed, trust the
   error message over the assistant, print things, and read the source.


## Part 2: Testing alternatives to GRPO

This part tells you what Part 2 asks of you, which scripts you run, the
checks every method must pass, and how it is submitted and read. The
mechanisms are taught in the second part of the RL for LLMs series (RFT
Alternative methods): RLLM-9 (self-training: self-distillation and RAFT),
RLLM-11 (on-policy distillation, OPD: the mechanism), RLLM-12 (the
teacher, and RLSD, which combines GRPO with self-distillation) and
RLLM-14 (how to check a comparison). RLLM-6 covers DAPO. This notebook
does not restate those mechanisms; where it needs a number or a flag it
says so.

**What Part 2 onward asks**: build and evaluate one or more alternatives
to GRPO -- cheaper, RL-adjacent methods -- against the exact same task
and the exact same held-out set as Part 1. Same rule as Part 1: **no
trained checkpoint is provided for any of these either.** You train
everything yourself, the same way you trained your GRPO checkpoint.

**What you already have, and should reuse**: your own Part 1
`evaluate.py`, if you built it as suggested (see Part 1's "Design note")
-- the same policy-loading/scoring function that produced your GRPO
money plot can score any new checkpoint you point it at.

**The specific methods that follow (OPD, self-distillation, RLSD) are
the ones this assignment guides you through in detail, not an exhaustive
list.** If you'd rather build and evaluate a different published
RL-adjacent method against this same task and held-out set -- instead
of, or in addition to, the ones here -- that's a legitimate way to
satisfy Part 2, provided your report explains what you chose and why,
and how you expect it to differ from plain GRPO going in, before you
have results. (One method this notebook doesn't walk you
through building: DAPO, a published refinement of GRPO itself rather
than an alternative family (RLLM-6) -- fair game here if you want to try it
instead of treating it as background reading only.)


### State your prediction before you run anything

For **every** method you build in Part 2 -- guided (OPD, self-distillation,
RLSD) or free-choice -- write down your prediction before you have
results, not after: what you expect its held-out accuracy and
collapse-behavior to look like relative to your Part 1 GRPO policy, and
*why*, grounded in how the method actually works (what it optimizes,
what it filters or doesn't, how it samples).

**The bar is "specific and falsifiable," not "correct."** "I expect it
to do about as well as GRPO" isn't gradeable -- there's no way to tell
whether you understood the method or got lucky. "I expect OPD to be more
prone to collapse than self-distillation, because self-distillation
discards low-reward samples before training on them and OPD trains on
everything it samples" is gradeable, and it's fine if it turns out wrong
-- a confidently wrong prediction with sound reasoning demonstrates real
understanding; a vague one doesn't, regardless of whether the guess
happens to land.

One more thing worth folding into "how it samples": every method here
exposes hyperparameters beyond the ones you've already tuned in Part 1
(a sampling temperature, and for the filtering methods, a reward
threshold deciding what counts as "good enough" to train on) -- these
aren't fixed defaults, and they can matter to a method's outcome. If a
filtering method's result looks surprising, or it seems to have very
little to actually filter, that's worth investigating as a hyperparameter
question before concluding the method itself doesn't work.

This is the same discipline Part 1 asked of you for $G$: evidence (we
suggested a readiness sweep), not a guess, because a $G$ picked from
intuition can silently produce a policy with no gradient signal to learn
from at all.
A hyperparameter copied from a default -- someone else's, another
method's, or the one this assignment happened to hand you -- is an
untested claim about your own setup, not a fact about it. Carry that
standard into the method you build here, not just the one Part 1
already asked a check about.


### Given vs. built (Part 2)

**Given, one complete training loop per guided method.** The
scripts are in the `part2_scripts/` directory of your bundle (they are not
part of `course_lib/`, which holds Part 1's given files); run them from the
bundle root so that `course_lib` imports as in "Set up the environment".
(Mirroring
`course_lib/grpo_loop.py`'s role in Part 1 -- you are not implementing
another training loop from scratch here; Part 1 already exercised that
skill, and repeating it three more times wouldn't teach anything new):

1. `self_distill.py` -- the full
   self-distillation loop (sample, filter, SFT, repeat). The same script
   also *is* RAFT via `--filter-mode topk` -- see "Given functions"
   below and RLLM-9 for why these two are one
   script with a different filter, not two.
2. `opd_train.py` -- the full
   on-policy distillation loop (sample from the student, score under a
   frozen teacher, forward-KL loss at every token).
3. `rlsd_train.py` -- the full
   RLSD orchestration loop ($K$ GRPO steps, one self-distillation
   consolidation round, repeat).

Each is a complete, runnable CLI script, same convention as Part 1's
`grpo_train.py`. **Be precise about what is and isn't seeded here**
(see "Training is itself random, even with a fixed seed" in Part 1's "Before you start" for why precision matters):
only `opd_train.py` has a `--seed` flag (it seeds the sampling and the
scenario choice, so a different value is a genuine replicate);
`self_distill.py` and `rlsd_train.py` expose none. RLSD's GRPO sub-phase
gets a real `torch.manual_seed` fix -- it inherits that from an
unmodified call into `course_lib.grpo_loop.run_grpo_stage`'s own default,
the same mechanism Part 1's `grpo_train.py` uses. `self_distill.py`'s own
sampling (and RLSD's self-distillation sub-phase, which reuses
`self_distill.py`'s logic) is genuinely unseeded -- run-to-run variation
there is expected stochasticity, not something you introduced by
touching a seed value, but it's also not the same "controlled, only
non-determinism can explain a gap" guarantee Part 1's GRPO stage gives
you. Keep that distinction in mind before concluding a surprising result
is either "a real hyperparameter effect" or "just noise" -- for these
methods, plain sampling variance is a third, live possibility.

**What's left for you to determine is NOT the mechanism** (RLLM-9, 11 and
12 teach that) **-- it's the same two things Part 1 already
taught you to determine rather than guess: the right hyperparameters for
your own checkpoint, and how to respond when a run collapses or
plateaus.** See "The plan" below.


### Given functions you'll call directly (Part 2)

1. `self_distill.py --sft-ckpt ... --rounds N --n-prompts N --g G
   --temperature T --reward-threshold X --filter-mode {threshold,topk}
   --ckpt-dir DIR` -- `filter-mode=threshold` (default) is
   self-distillation's pooled-threshold filter; `filter-mode=topk` is
   RAFT's per-prompt best-of-$G$ filter (`--topk` sets how many per
   prompt, RAFT uses 1). **The script ships a default
   `--reward-threshold` -- do not trust it for your own checkpoint
   without checking first** (see "The plan," step 2). A too-loose or
   too-strict threshold silently changes what this method can learn from,
   the same way an unchecked $G$ can in Part 1.
2. `opd_train.py --sft-ckpt ... --teacher MODEL --num-generations G
   --max-steps N --temperature T --learning-rate LR --ckpt-dir DIR` --
   **the OPD teacher is your own choice, passed with `--teacher`** (any
   frozen causal LM the script can load; RLLM-12 is about choosing it and
   about what the teacher can and cannot know). No teacher is provided. Also: `--num-
   generations` here is OPD's own per-step sample count, unrelated to
   GRPO's $G$ (OPD has no group-relative advantage to compute, so this
   number plays a different role -- RLLM-11 explains what it is).
3. `rlsd_train.py --sft-ckpt ... --cycles N --grpo-steps-per-cycle K
   --num-generations G --sd-n-prompts N --sd-g G --sd-temperature T --sd-reward-threshold X
   --ckpt-dir DIR` -- the `sd-*` flags configure RLSD's own internal
   self-distillation consolidation round; `--grpo-steps-per-cycle`
   ($K$, the number of GRPO steps in each cycle before the
   self-distillation round; RLLM-12 describes the cycle) is yours to
   choose, same calibration discipline as everything else here.
   `--num-generations` is the GRPO phase's $G$. If you omit it, the script
   silently reads `rl_readiness.json` (next to the checkpoint directory's
   parent) and otherwise falls back to `course_lib.grpo_loop`'s default
   $G$, so **pass your own measured $G$** from a readiness sweep on your
   checkpoint rather than trusting the fallback.
4. `course_lib.policy_checks.check_output_varies` (check A), plus
   `mean_rewards_from_rows` and `paired_significance` -- the same given
   functions as Part 1 (#12, #14, #15), reused unchanged (see "The
   collapse checks" below). `assert_money_plot_ordering` (check B) is not
   asserted for the alternative methods.


### The plan (Part 2)

1. **Pick the one method to build** -- exactly one of OPD,
   self-distillation, or RLSD (guided), or a free-choice published
   method (see the scope-setter above). Building more methods earns no
   extra required points. Write your prediction (previous
   section) before step 2.
2. **Calibrate the method's key hyperparameter(s) via a real, cheap
   diagnostic -- not a guess, and not the script's shipped default.**
   Same discipline Part 1 asked of you for $G$: an untested value copied
   from a default is an untested claim about your own setup, not a fact
   about it. Concretely, for a filtering method (self-distillation,
   RAFT, **and RLSD's own self-distillation sub-phase -- it filters too,
   via its own `--sd-reward-threshold`, and needs this same treatment,
   not just the temperature check below**): before running any training
   rounds, sample a batch of completions from your SFT checkpoint at
   your intended temperature, score them with the real reward function,
   and look at the resulting reward distribution -- what fraction clears
   each of several candidate threshold values? Pick a threshold from
   what you actually measured on *your* checkpoint. For OPD (and
   RLSD's GRPO sub-phase): temperature is inherited from GRPO's own
   default (1.0) everywhere in this module, never independently
   justified for these methods -- worth treating as your own thing to
   check rather than assuming it transfers.
3. **Run the given script.**
4. **A filtering method's keep-rate check (required for the filtering
   methods)**: if a round's keep rate
   comes back zero or vanishingly small, stop and report it as a finding,
   not a bug to silently patch around. (It is the analogue of Part 1's
   suggested exploration check, but required here because a zero-yield
   round trains on nothing.) What does it say about your calibrated
   threshold relative to your SFT checkpoint's actual strength? **If
   you're building RLSD, check its own self-distillation sub-phase's
   keep rate yourself** -- unlike standalone self-distillation/RAFT,
   RLSD doesn't hard-stop on a zero-yield consolidation round (it just
   falls back to the GRPO-only checkpoint for that cycle and continues),
   so a miscalibrated `--sd-reward-threshold` can pass silently instead
   of surfacing as an obvious failure.
5. **Evaluate**, reusing your Part 1 `evaluate.py` (see Part 1's "Design
   note" -- this is exactly why that note asked you to write it as a
   general, reusable function).
6. **Run the collapse checks below (checks A, C and, at least once, D)
   before trusting any accuracy number.**
7. **If your run collapses or plateaus well below your Part 1 GRPO
   baseline, diagnose why and try at least one concrete lever to escape
   it before reporting a final number** -- a different threshold/
   temperature calibration, more rounds/cycles, a different filter mode.
   Report both the original result and the escape attempt, whichever way
   it goes. (A reminder, our reading and not a best recipe: in our
   reference GRPO runs the reward sat near 0.5 until after step 400 to
   450 and then broke out or never did, so do not stop a run on reward
   level at a short step count; use the signal columns.) A documented, reasoned escape attempt -- successful or not --
   is worth more in the diagnosis bucket than a single collapsed number
   reported without investigation. (This is the Part 2 analogue of Part
   1's own required-figures rule: a run you diagnosed and responded to is
   a complete result, not something to hide.)


### Required figures (Part 2)

For the method you build:

1. **Your hyperparameter-calibration diagnostic** (step 2 above) -- a
   histogram or table showing measured yield/reward at several candidate
   values, with the value you picked and why. Same spirit as Part 1's
   readiness-sweep table (or whatever evidence you used to choose $G$), applied to whichever hyperparameter this
   method actually needs calibrated.
2. **The per-skill accuracy panel, extended** with your Part 2 method
   as additional rows/columns alongside Part 1's four policies.
3. **If you hit a collapse or a low-yield keep-rate check**: one
   diagnostic figure or log excerpt for it, and (if you attempted a fix)
   one showing the escape attempt's result.
4. **Your paired before/after check** (check D, below) for at least one comparison -- the two distributions
   you compared, side by side, and one sentence on what they do or don't
   show about whether the later policy actually moved toward the true
   distribution.


### The collapse checks

A trained policy can score well *on average* while quietly ignoring the
prompt -- one safe, generic answer can land close enough on many
scenarios to look competent in an aggregate number. The checks are named
A to D under "Checking whether a policy collapsed" in "Before you start".
For the method you build in Part 2, run **checks A and C**; run **check
D** at least once, on whichever comparison you find least convincing after
A and C. Check B (the ordering assertion) is Part 1's and is not asserted
here.

- **Check A, `course_lib.policy_checks.check_output_varies`** -- the same
  diversity check your Part 1 `evaluate.py` already runs on your GRPO
  policy (same `params_list` format, same `ranges`, `require_keys=("sigma",
  "r")`). Run it on your new policy too: greedy-decode a probe set
  spanning your scenario pool's extremes, and confirm the extracted
  $\sigma$/$r$ actually range across a meaningful fraction of their
  legal values, not a near-constant.
- **Check C, a scenario-blind baseline, built by you**: a synthetic policy
  that
  ignores the prompt entirely and always emits one fixed,
  correctly-formatted answer. Score it with the exact same reward
  function and held-out set as every real policy. If a trained policy's
  aggregate performance *matches* this baseline closely, that's
  independent confirmation it collapsed -- it isn't doing anything a
  policy that never reads the prompt couldn't already do.
- **Check D, a paired before/after check, built by you.** Check A
  looks at one policy's snapshot in isolation -- diverse enough or not,
  right now. It cannot tell you *why* a number looks the way it does,
  only that it does. That gap is not hypothetical: a real checkpoint from
  this course had one extracted key at the *same constant on 90% of
  held-out probes*, yet the aggregate check passed, because a different
  key's healthy diversity kept the joint uniqueness count high and the
  collapsed key's own min-max span was inflated by the unaffected 10%'s
  spread. The fix (the per-key majority-share statistic behind
  `require_keys`) closes that specific hole, but "does this snapshot look
  diverse enough" is structurally a different question from "did this
  key's distribution *actually move* toward the true one, or did the
  accuracy gain come from recentering to a different, still-uninformative
  constant." Build a check that answers the second question for at least
  one Part 2 comparison: take two policies' decoded outputs on the *same*
  held-out probe set (e.g. your SFT-only policy vs. a trained one),
  extract the same key(s) from both, and compare the two distributions
  directly -- not just each one's own diversity, but whether the second
  one's mass sits somewhere meaningfully different from the first's. Use
  this as a second opinion whenever check A passes but you're not fully
  convinced.

Decode greedily for checks A, C and D, matching real evaluation -- a
temperature-sampled probe can look diverse from a collapsed policy's own
distribution tails even after its true (greedy) mode has already
collapsed to one answer.


### Is the gap real? Checking significance before you compare

Once a new method's policy clears the collapse checks above,
you'll have two mean rewards to compare -- say, GRPO's and this method's,
both scored on the same held-out set. **A gap between two means is not
automatically real.** With a few-hundred-scenario held-out set and a
reward that's bounded and somewhat lumpy (not a smooth bell curve), a
modest difference can plausibly be sampling noise.

`course_lib.policy_checks.paired_significance(rows, policy_a, policy_b)`
(given function #15) -- takes the same tidy `(scenario_id, policy, reward)` rows your money
plot already uses (plain tuples, one per scenario and policy, for example
`("s017", "grpo", 0.82)` and `("s017", "sft", 0.64)`), pairs by `scenario_id` (both policies were scored on
the *same* scenarios, so pair rather than treat them as independent
samples), and returns `{"mean_diff", "t_stat", "ci_95": (lo, hi),
"n_wins", "n_losses", "n_ties"}` -- the CI comes from a bootstrap
(resample the paired differences with replacement, repeat thousands of
times, take the 2.5th/97.5th percentiles of the resampled means), not a
textbook formula, since it doesn't require assuming the differences are
normally distributed.

**Run this on any comparison your report claims is a real improvement**
(Part 2's method vs. GRPO, or vs. SFT-only) -- a CI that excludes zero
supports the claim; one that straddles zero means the claim isn't
actually backed by your own data, whatever the two raw means happen to
say. Report the win/loss/tie counts alongside the CI too: "real" and
"landslide" are different claims, and a report should only make the one
its numbers actually support.

This only applies once you have two non-collapsed policies to compare --
if your method's policy fails the collapse checks above, the
comparison is moot; report the collapse instead of a significance test
on a policy that isn't doing anything conditional on the prompt.


### Summarize across methods, in one table

Once you've tested your method, build a single summary
table: one row for your chosen method plus one for your Part 1 GRPO policy as a
baseline (and one per extra method, if you built any), one column for held-out accuracy. **Add whatever other
columns you think are informative, and justify each one in a sentence** --
this assignment isn't going to hand you the list; Part 2's own checks
above should already suggest some candidates.

Then discuss the table in your report. A useful test of whether the
table itself is complete: if your discussion leans on a signal that
isn't a column in the table, that signal probably belongs in the table.


### Part 2: how to submit and how it is graded

**Submit** a second archive, `part2.tar.gz`, whose top level is
`report/part2/` and `code/` (same layout idea as the Part 1 archive):

```
part2.tar.gz
  code/                     # your Part 2 code (see below)
  report/part2/
    report.md                 # headings below, spelled exactly
    summary_table.csv         # the across-methods table (one row per method, GRPO baseline included)
    accuracy_panel_part2.csv  # same schema as Part 1's accuracy_panel.csv, with your methods' policy names added
    accuracy_panel_part2.png  # the extended per-skill panel figure, with your methods added
    calibration.png           # the hyperparameter-calibration figure or table (plan step 2)
    check_d.png               # check D: the side-by-side figure on the comparison you find least convincing
    escape_attempt.png        # IF A RUN COLLAPSED OR PLATEAUED: the figure for your escape attempt
    metrics/                  # one log per method run (what the given scripts write), plus the paired-check data
    environment.txt           # pip freeze, as in Part 1
```

`report.md` headings: `## Prediction` (written before any result, one
per method; its FIRST line must read `Chosen method: <name>`, and that is the
method graded on the required Part 2 lines; this is a *line*, not a
separate heading), `## Calibration`, `## Collapse checks` (A, C, D for each
method), `## Significance` (the `paired_significance` output for each
claimed improvement), `## Summary table`, `## Escape attempt` (if a run
collapsed or plateaued; write "None" otherwise).

Put your Part 2 source code (the scripts you ran with your flags, any
driver, your check D and check C code) in the top-level `code/` directory of
`part2.tar.gz`, next to `report/` (the scripts themselves are in `part2_scripts/` in your bundle;
your code is whatever you changed or added around them).

**How Part 2 is graded: 50 points, all required.** Exactly one method is
required; building more methods earns no extra required points. A second
method counts as an Extra only if it adds a new, unguided hypothesis or
comparison. Each line is scored full, partial or none
by what your report and files show. An honest negative result (your method
loses to GRPO, or collapses) earns full credit when it is well evidenced.

| Line | Points | What earns it |
|---|---|---|
| Prediction before results | 10 | for your method, a specific, falsifiable prediction, written before the results, with reasoning from how the method works |
| Calibration of the method's key hyperparameter | 10 | a real diagnostic (a threshold or temperature table or histogram, with the keep-rate check) and a value chosen from it with a stated reason, not the shipped default |
| Collapse checks | 15 | checks A and C run and reported for your method, and check D at least once on the comparison you find least convincing |
| Significance and the summary table | 15 | `paired_significance` on each claimed improvement with the CI and win/loss/tie counts; a summary table (your method, the GRPO baseline, any extra method) with justified columns; claims that match the numbers |


## Part 3: Build your own SFT data, in stages

Everything through Part 2 used `sft_examples.py`'s given training rows.
This part asks you to look at those rows critically, then build your own
from scratch -- **only after** you've completed Part 2, so you have a
real GRPO run and a Part 2 alternative to reason from, not just a
blank-page format choice. **Part 3 is required (50 points)**, and **completing more stages earns
more points**. It is organized as four stages, and Stages 1 and 2
each end in a **recorded decision** about whether to continue (Stage 0 and
Stage 3 do not: Stage 3 is the final write-up), because the later stages
cost real GPU time. Each stage costs GPU time, and free
Colab rations GPU, so choose how far to go knowing the credit (table
below).

**What each stage costs.** These are times we measured on our own T4 runs
(free Colab may differ, and it rations GPU: heavy use in a day can mean no
GPU the next day, so plan your sessions):

| Stage | What | Our measured cost on a T4 |
|---|---|---|
| 0 | critique the given data, state a prediction | no compute |
| 1 | build your SFT data, retrain SFT, compare with the given-data SFT checkpoint | SFT is minutes; evaluation is cheap |
| 2 | pilot: optional readiness sweep for $G$, then the signal stage (3b, about 150 steps) | about 0.5 h |
| 3 | the full run (3c, 800 steps) and evaluation of the policies on 200 held-out scenarios | about 2.5 to 3 h for the run, about 1 h to evaluate four policies; a whole Part 1 pipeline was about 5 to 6 h |

**Stopping at a decision point is a legitimate outcome.** If you stop at one with
evidence and a clear statement of what you would have done next, you earn
credit for the stages you reached (an honest, evidence-based stop at the
Stage 2 gate earns 30 of the 50 points). Stage 3 is optional *by decision*,
not by default: you decide at the Stage 2 decision point, with your own evidence.

**A caution about stopping rules (our reading, not a best recipe).** Our
reference GRPO runs sat on a reward plateau near 0.5 until after step 400
to 450, and then either broke out (to 0.76 to 0.88) or never did; no
training-log statistic at step 200, 400 or 600 separated the two kinds of
run. So **do not stop a run on reward level at a short step count.** The
signal columns (`within_group_std`, `frac_all_tied`, `parse_rate`, KL) are
the better evidence for a pilot, and even they are a judgment, not a rule:
state your criterion beforehand and say what you could not rule out.


### Stage 0: critique the given data and predict (before anything runs)

**Look critically at what you were given.** Read `sft_examples.py`'s
source, not just its docstring. Is there anything about its completion
format, its noising strategy, or its structure that you'd have designed
differently? You don't need a strong opinion here to move on to the next
stage -- but if you have one, state it up front, before you build
anything, not reverse-engineered afterward to match whatever results you
get.

**State your design choice and a prediction.** Say what your own SFT data
will do differently, what you expect to change (SFT-only accuracy, the
collapse checks, whether GRPO will start from a better place) and why.
**Specific and falsifiable, as in Part 2:** a confidently wrong prediction
with sound reasoning is worth more than a vague correct one.


### Stage 1: build your own SFT data, retrain, compare, decide

**Build your own.** You're free to construct your own SFT training
examples in **any reasonable format** -- this is genuinely open, not a
guided rebuild of `sft_examples.py` with one knob changed. One
constraint, not a suggestion: your completions still have to be
scoreable by the given, unmodified `reward.py` -- it isn't yours to
tune (same rule as Part 1), which means your format still needs to
produce the seven canonical `` name = `expr` `` lines somewhere the
harness can parse them, however you choose to surround them.

Retrain with `sft_train.py` against your own examples, then compare the new
SFT checkpoint with your Part 1 (given-data) SFT checkpoint **on the same
held-out set**, and run collapse checks A and C on it. This stage is cheap
(CPU or a short GPU session).

**The standard your own design has to meet (or deliberately break).**
Part 1's given examples aren't just avoiding leaked *values* -- they're
avoiding teaching the *skill* itself. Every completion states a
plausible-but-wrong $\sigma$/$r$, so SFT only ever teaches your model the
completion *shape* (the `<think>`/`<answer>` structure, the `` name =
`expr` `` lines), never a route to the actual extraction accuracy that
Part 1's GRPO stage is supposed to be the one thing that teaches. That's
the whole point of comparing `sft_only` against `grpo` in your accuracy
panel -- if your own design lets real extraction skill leak in through
some other channel (e.g., completions built from `rule_extract.py`'s own
correct output, even without literally emitting the true numeric token),
you've quietly moved the thing Part 1's GRPO stage is supposed to
demonstrate into the SFT stage, and the comparison stops meaning what the assignment claims it
means. You don't have to preserve this property -- deliberately testing
what happens if SFT *does* partially teach extraction is a legitimate
direction -- but if you break it, that's a hypothesis to state up
front (the Stage 0 rule), not a side effect to discover in your own
numbers after the fact.

The two rules this reintroduces, now that you're authoring SFT data
yourself:

1. **SFT audit rule**: audit your generated rows so that no numeric
   token in any completion coincidentally matches the scenario's *true*
   $\sigma$/$r$ or price -- same rule Part 1's given examples already
   satisfy, now yours to enforce.
2. **`<think>`-block content is your call, but state it.** Part 1's given
   examples happen to contain nothing but the seven derivation lines
   inside `<think>` -- Part 3 doesn't require you to keep that property.
   If you change it, say so explicitly in your report, and say why.
   `course_lib.bs_reasoning.sft_format_check.check_no_preamble` (given
   function #13, Part 1) is available as a diagnostic either way -- to
   confirm you kept the given property, or to confirm exactly how and
   where your own design departs from it.

**Decision record 1:** *is the new SFT checkpoint good enough to start
GRPO?* (Same four fields as Part 1's decisions; see below.) "No" is a
fine answer if your evidence supports it: fix the data, or stop here.


### Stage 2: pilot (about 0.5 h), then decide

If you judge it needed, run a readiness sweep for $G$ on the new
checkpoint (your SFT data changed what the model samples, so a $G$ chosen
for the old checkpoint is an untested claim here). Then run the signal
stage (the 3b run of Part 1, about 150 steps) from the new checkpoint.

**Decision record 2:** *continue to the full run, stop, or change
something?* State the criterion **beforehand** and apply it to evidence
from your logs. Remember the plateau caution above: reward level at 150
steps is **not** a stop rule; use the signal columns and say what they do
and do not tell you.


### Stage 3: the full run (about 3 h), optional by decision

If your Stage 2 decision was to continue, run the full stage (3c, 800
steps) from your new checkpoint and evaluate. Compare your new pipeline
with your Part 1 pipeline on the same held-out set, using
`paired_significance` on every comparison you claim is an improvement
(CI and win/loss/tie counts), plus collapse checks A and C. Then write up
what happened against your Stage 0 prediction.

If you decided not to run Stage 3, say so, say why from your evidence,
and say what you expect would have happened and what would have changed
your mind. That is a complete Stage 3.


### Part 3: how to submit and how it is graded

**Submit** inside your main `part1.tar.gz`: a `## Part 3` section in
`report.md`, with these sub-headings, spelled exactly:

```
## Part 3
### Stage 0
### Stage 1
### Stage 2
### Stage 3
```

You submit Part 3 inside `part1.tar.gz` (Part 3 extends your Part 1
pipeline, so it shares the one `report.md` with Part 1 (its logs go in
`report/part3/metrics/`); the
archive layout is defined once, under "How to submit Part 1"). The Part 3
files go in `report/part3/` of that archive:

```
report/part3/
  sft_data_generator.py     # your SFT data generator (also copy it into code/)
  sft_data_sample.jsonl     # a sample of its rows
  sft_eval.md               # new checkpoint: evaluation, checks A and C output, comparison with the given-data SFT checkpoint
  figures/                  # Part 3 figures, referenced from report.md
  metrics/                  # Part 3 pilot and full-run logs (jsonl), e.g. grpo_signal_p3.jsonl, grpo_full_p3.jsonl
  panel_part3.csv           # IF YOU RAN STAGE 3: accuracy panel
  paired_significance.txt   # IF YOU RAN STAGE 3: the paired_significance output
```

Part 3 source code goes in the top-level `code/` directory, as in Part 1.

Under each of `### Stage 1` and `### Stage 2` give the four-field decision
record Part 1 uses (**Decision**, **Criterion**, **Evidence**,
**Outcome**; one per line, bold labels, as in "Pre-flight and decisions"),
and a `**When:**` line saying whether you decided before or after seeing
the numbers. Stage 3 has no decision record: under `### Stage 3` give the comparison
or, if you did not run it, the explanation.

**Part 3 is worth 50 points.** Each stage has *work* points (you did the
stage) and, for Stages 1 and 2, *decision* points (the quality of the
continue-or-stop decision you recorded). Decision points are earned at the
stage where you stop too.

| Stage | Points | What earns it |
|---|---|---|
| Stage 0: design critique and prediction | 6 | a critique that comes from reading the source, a stated design choice, and a specific falsifiable prediction written before results |
| Stage 1: data, checkpoint, comparison | 12 = work 8 + decision 4 | work: your own data (audited), the retrained checkpoint, the same-held-out-set comparison with the given-data SFT checkpoint, checks A and C; decision: record 1 |
| Stage 2: pilot signal stage | 12 = work 6 + decision 6 | work: the pilot evidence from your logs; decision: record 2, with a criterion stated beforehand and a decision that follows from the evidence |
| Stage 3: full run, evaluation, paired comparison | 20 = work 12 + analysis 8 | work: the full run, the evaluation and the `paired_significance` comparison; analysis: the write-up against your Stage 0 prediction |

An honest, evidence-based stop at the Stage 2 decision point earns Stages 0,
1 and 2 (30 of the 50); completing Stage 3 earns the rest. A stop rule that
rests on reward level at a short step count earns only half of that stage's
decision points (see the plateau caution above).

Not reaching Stage 3 with an evidence-based stopping decision costs only the Stage 3 points. Pairs naturally with a different training algorithm from
Part 2 (DAPO included) -- but that would be your own Extra, not a Part 3
requirement.


## Extras: the "above and beyond" third

Two thirds of the grade comes from what we specifically ask for in Parts 1
to 3. The last third is for ideas we do **not** guide you on. You choose
them; there is no list to complete. A report with no `## Extras`
section simply has 0 here, with no penalty.

**What an extra is.** An idea of your own: a new hypothesis, comparison or
analysis that the required parts do not ask for.

**What an extra is not.**

- Not anything we guided you through (the required items and the suggested
  checks).
- Not "run another seed of what we asked".
- Not an ablation of something we suggested, unless it adds a new
  prediction, comparison or analysis of its own.

**How an extra is graded.** Four criteria:

1. **Novelty**: how much it is your own idea, beyond the guided work.
2. **Importance**: why the question matters (to this task, or to
   post-training in general).
3. **Explained rationale**: a hypothesis, stated *before* you run it,
   with the reason you expect it.
4. **Good experimental process**: a controlled comparison, a prediction
   written first, honest reporting, and files that let a reader check.

**An experiment that fails, or refutes your hypothesis, but is well
designed earns a lot of credit** -- at least a third to a half of the
item's range for process and rationale alone. A flashy result with a
careless design does not.

**Points scale with difficulty, relative to the effort of Part 1.** A
small extra is roughly a tenth of Part 1's effort, a medium one a fifth,
a large one a third or more:

| Size | Rough effort (relative to Part 1) | Points per extra |
|---|---|---|
| Small | about a tenth | up to about 10 |
| Medium | about a fifth | up to about 20 |
| Large | a third or more | up to about 35 |

**How to submit.** Add a **`## Extras`** section to your Part 1
`report.md`. For each extra: *what* you did, *why it is new*, *the
hypothesis you stated beforehand and why*, *the effort* it took, and *the
evidence* (files and numbers). Any file an extra relies on (a figure, a
log, code) must be in your submission or the extra cannot be credited.
Any original idea is welcome; we do not give a list. Doing the pre-flight
*before* the full run earns a small bonus of up to 10 points, counted
inside the 100.
